In [29]:
# CELL 1 - INSTALL
# Installs the three libraries this notebook needs:
# box2d (the physics engine LunarLander runs on), stable-baselines3 (the PPO
# training library) and imageio (turns pictures into GIFs).
!pip install box2d stable-baselines3 imageio

In [30]:
# CELL 2 - IMPORTS AND FOLDERS
# Loads the libraries into Python and creates three folders for our outputs:
# models/ (trained agents), results/ (tables) and media/ (GIFs).
# Everything is saved under /kaggle/working so Kaggle keeps it.
import os, time, shutil
import numpy as np, pandas as pd
import gymnasium as gym, imageio
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

MODELS = "/kaggle/working/models"
RESULTS = "/kaggle/working/results"
MEDIA = "/kaggle/working/media"
for folder in (MODELS, RESULTS, MEDIA):
    os.makedirs(folder, exist_ok=True)   # create the folder if it doesn't exist

In [31]:
# CELL 3 - COPY THE BASELINE AGENTS
# The 5 baseline agents were trained in an earlier notebook (added here as an input).
# Input folders are read-only, so we copy the agents into this notebook's models/ folder
# where the evaluation code can load them.
shutil.copytree("/kaggle/input/notebooks/toyibatadele/lunarlander-ppo/models", MODELS, dirs_exist_ok=True)
print(os.listdir(MODELS))   # should list the 5 ppo_baseline_seed*.zip files

['custom.css', 'ppo_baseline_seed0.zip', 'ppo_baseline_seed2.zip', '__output__.json', 'ppo_baseline_seed4.zip', 'results', 'ppo_baseline_seed3.zip', '__results__.html', 'media', 'ppo_baseline_seed1.zip', '__notebook__.ipynb', 'models']


In [32]:
# CELL 4 - PPO SETTINGS
# The training settings (hyperparameters) for every agent, from the RL Baselines3 Zoo
# model card for LunarLander. They are identical for all agents, so the ONLY thing
# that differs between the baseline, broken and fixed agents is the reward.
HYPERPARAMS = dict(n_steps=1024, batch_size=64, n_epochs=4,
                   gamma=0.999, gae_lambda=0.98, ent_coef=0.01)

In [33]:
# CELL 5 - TRAINING FUNCTION
# Defines train(): it trains one PPO agent for 1,000,000 steps and saves it.
# If a reward wrapper is given, the agent learns from the changed reward.
# Defining the function does not train anything; training happens in cell 12.
def train(name, seed, wrapper_class=None, wrapper_kwargs=None, steps=1_000_000):
    # Skip training if this agent is already saved
    if os.path.exists(f"{MODELS}/{name}.zip"):
        print(f"{name}: already trained, skipping")
        return
    start = time.time()
    # Run 16 copies of the game at once; apply the reward wrapper if one is given
    env = make_vec_env("LunarLander-v3", n_envs=16, seed=seed,
                       wrapper_class=wrapper_class, wrapper_kwargs=wrapper_kwargs)
    # Create the PPO agent (a small neural network on CPU)
    model = PPO("MlpPolicy", env, seed=seed, device="cpu", verbose=0, **HYPERPARAMS)
    model.learn(total_timesteps=steps)   # train for 1,000,000 steps
    model.save(f"{MODELS}/{name}")       # save the trained agent
    print(f"{name}: trained in {(time.time()-start)/60:.1f} min")

In [34]:
# CELL 6 - EVALUATION FUNCTION
# Defines evaluate(): it loads a saved agent and plays 100 games with it.
# Scoring always uses the ORIGINAL game reward, so every agent is judged by the same rule.
# It records each game's score, length and how it ended (landed, crash or timeout),
# and saves the results as a CSV file.
def evaluate(name, n_episodes=100, first_seed=1000):
    model = PPO.load(f"{MODELS}/{name}", device="cpu")   # load the saved agent
    env = gym.make("LunarLander-v3")   # plain game, so the score is always the ORIGINAL reward
    rows = []
    for i in range(n_episodes):
        # Each game gets its own fixed seed, so every agent faces the same starting conditions
        obs, _ = env.reset(seed=first_seed + i)
        total, steps, over = 0.0, 0, False
        while not over:
            # deterministic=True: the agent always picks its best action
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(int(action))
            total += reward   # add up the reward for this game
            steps += 1
            over = terminated or truncated   # the game ends on a crash, landing or time limit
        # Work out how the game ended, using the last step's reward
        if truncated:
            outcome = "timeout"    # ran out of time
        elif reward == -100:
            outcome = "crash"
        elif reward == 100:
            outcome = "landed"
        else:
            outcome = "other"
        rows.append(dict(model=name, episode=i, reward=total, steps=steps, outcome=outcome))
    env.close()
    df = pd.DataFrame(rows)   # one row per game
    df.to_csv(f"{RESULTS}/eval_{name}.csv", index=False)
    return df

In [35]:
# CELL 7 - SUMMARY FUNCTION
# Defines summarize(): it turns the 100-game results from evaluate() into the
# numbers we report: average score, spread, best and worst game, success rate,
# and the share of games that landed, crashed or timed out.
def summarize(df):
    return dict(mean=df["reward"].mean(),                            # average score
                std=df["reward"].std(),                              # how much scores vary
                min=df["reward"].min(),                              # worst game
                max=df["reward"].max(),                              # best game
                success=(df["reward"] >= 200).mean() * 100,          # % of games scoring 200+
                landed=(df["outcome"] == "landed").mean() * 100,     # % that ended in a landing
                crash=(df["outcome"] == "crash").mean() * 100,       # % that ended in a crash
                timeout=(df["outcome"] == "timeout").mean() * 100,   # % that ran out of time
                avg_steps=df["steps"].mean())                        # average game length

In [36]:
# CELL 8 - GIF FUNCTION
# Defines record_gif(): it plays one game with a saved agent, takes a picture
# after every step, and joins the pictures into a GIF in the media/ folder.
# Every agent uses the same starting seed, so the GIFs can be compared side by side.
def record_gif(name, gif_name, seed=1000):
    model = PPO.load(f"{MODELS}/{name}", device="cpu")
    env = gym.make("LunarLander-v3", render_mode="rgb_array")   # draw pictures into memory
    obs, _ = env.reset(seed=seed)   # same seed every time, so GIFs are comparable
    frames = [env.render()]         # save the first picture
    over, steps = False, 0
    while not over and steps < 1000:   # stop after 1000 steps at most
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        frames.append(env.render())    # save a picture after every step
        over = terminated or truncated
        steps += 1
    env.close()
    imageio.mimsave(f"{MEDIA}/{gif_name}.gif", frames, fps=30)   # join the pictures into a GIF
    print(f"saved {gif_name}.gif with {len(frames)} frames")

In [37]:
# CELL 9 - BEHAVIOUR CHECK FUNCTION
# Defines profile(): it plays 30 games and measures HOW the agent flies, not its score:
# game length, how often an engine fires, how often the button changes (switches),
# how often it alternates between left and right engines (lr_flips), and average tilt.
# This is how we check whether a reward change actually made the flying smoother.
def profile(name, n_episodes=30, first_seed=1000):
    model = PPO.load(f"{MODELS}/{name}", device="cpu")
    env = gym.make("LunarLander-v3")
    rows = []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=first_seed + i)
        steps = switches = lr_flips = engine_steps = 0
        angle_sum, last, last_side, over = 0.0, None, None, False
        while not over:
            action, _ = model.predict(obs, deterministic=True)
            action = int(action)
            if action != 0:
                engine_steps += 1                      # an engine is firing this step
            if last is not None and action != last:
                switches += 1                          # the button changed from last step
            if action in (1, 3):                       # a side engine (left or right)
                if last_side is not None and action != last_side:
                    lr_flips += 1                      # switched sides (counts even with a pause between)
                last_side = action
            last = action
            obs, reward, terminated, truncated, _ = env.step(action)
            angle_sum += abs(obs[4])                   # add up the tilt (slot 4 is the angle)
            steps += 1
            over = terminated or truncated
        rows.append(dict(steps=steps, engine_steps=engine_steps, switches=switches,
                         lr_flips=lr_flips, mean_abs_angle=angle_sum / steps))
    env.close()
    return pd.DataFrame(rows).mean().round(2)   # average over all the games

In [38]:
# CELL 10 - THE REWARD WRAPPER
# Defines Smooth, a wrapper: a layer around the game that changes the reward
# before the agent sees it. It has two modes:
#   "penalty"    = Rule B, the fix: subtract a little whenever the agent flips
#                  directly between the left (1) and right (3) engines.
#   "hold_bonus" = the deliberately broken version: add a bonus on every step
#                  the agent presses the same button as the step before.
class Smooth(gym.Wrapper):
    def __init__(self, env, mode="penalty", coef=0.05):
        super().__init__(env)
        self.mode, self.coef = mode, coef   # which rule to use, and how strong it is
        self.last_action = None             # memory of the previous button press

    def reset(self, **kwargs):
        self.last_action = None             # new game: forget the previous game's last button
        return self.env.reset(**kwargs)

    def step(self, action):
        # Pass the button press to the real game and get its normal result
        obs, reward, terminated, truncated, info = self.env.step(action)
        action = int(action)
        bonus = 0.0                         # extra reward to add; zero unless a rule applies
        if self.last_action is not None:    # skip the very first step of a game
            if self.mode == "penalty":
                # Rule B (the fix): penalise flipping between left (1) and right (3)
                if {action, self.last_action} == {1, 3}:
                    bonus = -self.coef
            elif self.mode == "hold_bonus":
                # The broken version: pay a bonus every step the same button is repeated
                if action == self.last_action:
                    bonus = self.coef
        self.last_action = action           # remember this button for the next step
        info["original_reward"] = reward    # keep the untouched reward for checking
        info["bonus"] = bonus               # keep the extra reward for checking
        return obs, reward + bonus, terminated, truncated, info   # agent sees reward + bonus

In [39]:
# CELL 11 - WRAPPER TEST
# Checks the wrapper works before spending an hour on training.
# Plays one game with random button presses in each mode and counts the bonus.
# Expected: penalty gives a negative total, hold_bonus a positive one (fires ~25% of steps).
for mode in ["penalty", "hold_bonus"]:
    env = Smooth(gym.make("LunarLander-v3"), mode=mode, coef=0.05)
    env.reset(seed=0)
    total_bonus, bonus_steps, step, over = 0.0, 0, 0, False
    while not over:
        obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
        total_bonus += info["bonus"]          # running total of the extra reward
        bonus_steps += info["bonus"] != 0     # count steps where a bonus or penalty applied
        step += 1
        over = terminated or truncated
    print(mode, "| steps:", step, "| steps with a bonus:", bonus_steps, "| total bonus:", round(total_bonus, 2))

penalty | steps: 73 | steps with a bonus: 11 | total bonus: -0.55
hold_bonus | steps: 83 | steps with a bonus: 27 | total bonus: 1.35


In [40]:
# CELL 12 - TRAIN THE NEW AGENTS (slow: about 60-70 minutes, run with Save & Run All)
# Trains three agents with the same settings as the baseline; only the reward differs.
# Broken: +0.5 bonus on every step the same button is repeated.
train("broken_holdbonus_seed0", 0, Smooth, {"mode": "hold_bonus", "coef": 0.5})
# Fixed: -0.05 penalty for flipping between left and right, trained on two seeds.
train("fixed_penalty_seed0", 0, Smooth, {"mode": "penalty", "coef": 0.05})
train("fixed_penalty_seed1", 1, Smooth, {"mode": "penalty", "coef": 0.05})

broken_holdbonus_seed0: trained in 26.1 min
fixed_penalty_seed0: trained in 20.3 min
fixed_penalty_seed1: trained in 20.9 min


In [41]:
# CELL 13 - COMPARISON TABLE (only works after cell 12 has finished)
# Plays 100 games with the baseline, the broken agent and the two fixed agents,
# all scored on the original reward, and saves one comparison table.
names = ["ppo_baseline_seed1", "broken_holdbonus_seed0", "fixed_penalty_seed0", "fixed_penalty_seed1"]
rows = []
for n in names:
    df = evaluate(n)                           # play 100 games
    rows.append({"model": n, **summarize(df)}) # keep the summary numbers
table = pd.DataFrame(rows).round(1)
print(table.to_string(index=False))
table.to_csv(f"{RESULTS}/comparison.csv", index=False)

                 model  mean  std   min   max  success  landed  crash  timeout  avg_steps
    ppo_baseline_seed1 255.2 20.2 209.6 294.0    100.0   100.0    0.0      0.0      365.4
broken_holdbonus_seed0 244.2 35.7  33.3 293.2     96.0    97.0    1.0      2.0      383.3
   fixed_penalty_seed0 252.8 30.7  40.4 295.7     98.0    99.0    1.0      0.0      336.7
   fixed_penalty_seed1 257.4 19.5 210.8 299.5    100.0   100.0    0.0      0.0      358.7


In [42]:
# CELL 14 - BEHAVIOUR TABLE AND GIFS (only works after cell 12 has finished)
# Measures how each agent flies (switches, left/right flips, engine use, tilt),
# saves that as a table, and records one GIF per agent from the same start.
prof = pd.DataFrame({n: profile(n) for n in names}).T
print(prof)
prof.to_csv(f"{RESULTS}/behaviour.csv")

for n in names:
    record_gif(n, n)

                         steps  engine_steps  switches  lr_flips  \
ppo_baseline_seed1      368.07        243.23    228.00     44.23   
broken_holdbonus_seed0  391.23        331.87    246.37    103.07   
fixed_penalty_seed0     343.07        219.07    202.03     32.60   
fixed_penalty_seed1     359.73        204.53    216.63     14.20   

                        mean_abs_angle  
ppo_baseline_seed1                0.06  
broken_holdbonus_seed0            0.05  
fixed_penalty_seed0               0.07  
fixed_penalty_seed1               0.06  
saved ppo_baseline_seed1.gif with 400 frames
saved broken_holdbonus_seed0.gif with 395 frames
saved fixed_penalty_seed0.gif with 352 frames
saved fixed_penalty_seed1.gif with 375 frames


In [44]:
# CELL 15 - FAILURE GIF
# Records game 27 (seed 1027), one of the broken agent's timed-out games, as a GIF.
record_gif("broken_holdbonus_seed0", "broken_failure", seed=1027)

results found: []
model found: []


IndexError: list index out of range